<a href="https://colab.research.google.com/github/hcopley/st_542_ml_systematic_review/blob/main/03_modeling/SuperLearner_in_parallel_with_folds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

install.packages(
c('SuperLearner'
,'doParallel'
,'arm'
,'glmnet'
,'ranger')
)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘bitops’, ‘gtools’, ‘caTools’, ‘gplots’, ‘rbibutils’, ‘ROCR’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘nnls’, ‘gam’, ‘cvAUC’, ‘foreach’, ‘iterators’, ‘lme4’, ‘abind’, ‘coda’, ‘shape’, ‘RcppEigen’




In [2]:
# Load all required libraries after installation
library(tidyverse)
library(SuperLearner)
library(data.table)
library(parallel)
library(doParallel)
library(arm)
library(glmnet)
library(ranger)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: nnls

Loading required package: gam

Loading required package: splines

Loading required package: foreach


Attaching package: ‘foreach’


The following objects are masked from ‘package:purrr’:

    accumulate, when


Loaded gam 1.22-7


Super Learner

Version: 2.0-40

Package created on 2025-12-14



Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, isoyear, mday, minute, month, quar

In [3]:
dt_tfidf <- readRDS("tfidf_svd_features.rds")
dt_doc2vec <- readRDS('doc2vec_features.rds' )
dt_pubmed <- readRDS('pubmedbert_embeddings.rds')

#load in the folds created in the data preparation step.
cv_folds <- readRDS("cv_folds.rds")

In [4]:
superlearner_parallel_pipeline <- function(.data, cv_folds, run_test_sample = FALSE, test_sample_n_cols = 25) {

  #stop the previous cluster just in case it is still running from a previous run of the notebook.
  #This prevents errors about clusters already running and ensures a clean start for parallel processing.
 if(exists("cl")) {
  suppressWarnings(stopCluster(cl))
 }

  # Separate the response from features
  Y <- .data[[2]]
  X <- .data[, -c(1, 2)]

  if(run_test_sample ==TRUE) {
    # For quick testing, use a smaller sample of columns and rows. This is not for final results but can speed up iterations during development.
    set.seed(542)
    sample_columns <- sample(ncol(X), min(test_sample_n_cols, ncol(X)))

    X <- X[, sample_columns]
  }

  # Ensure that the response is mumeric
  Y <- as.numeric(Y)

  # Compute class weights
  pos_rate <- mean(Y == 1)
  neg_rate <- mean(Y == 0)

  # Assign inverse class frequency weights to balance the classes
  obs_weights <- ifelse(Y == 1, 1/pos_rate, 1/neg_rate)

  #set the number of cores for parallel processing. We use one less than the total to avoid overloading the system.
cl <- makeCluster(detectCores() - 1)
#set up parallel backend to use the cluster
registerDoParallel(cl)
#set a seed on each worker to ensure reproducibility across parallel processes
clusterSetRNGStream(cl, iseed = 42)
#send SuperLearner to the cluster workers so they can run the learners in parallel
clusterEvalQ(cl, {library(SuperLearner)})
#send all necessary objects to each cluster worker so they can run the learners in parallel without errors
#the envir = environment() argument ensures that the objects are taken from the current environment of the function
#which is important to avoid issues with object not found errors in the workers as the default is for clusterExport to look in the global environment which may not have the necessary objects when running inside a function.xpor
clusterExport(cl, list(
  "Y", "X", "learners", "obs_weights", "cv_folds"
), envir = environment())

sl_cv <- CV.SuperLearner(
  Y = Y,
  X = X,
  SL.library = learners,
  family = binomial(),
  obsWeights = obs_weights,
  method = "method.NNLS",
  cvControl = list(V = 5, validRows = cv_folds), # when passing the folds set V here to the number of folds expected
  parallel = cl
)

stopCluster(cl)

return(sl_cv)

}

In [5]:
learners <- c(
  "SL.glmnet",        # Elastic-net logistic (BEST for TF-IDF)
  "SL.xgboost",       # Shallow boosted trees
 "SL.ranger",
 "SL.ksvm",
 "SL.bayesglm"
)

In [ ]:
pubmed_res <- superlearner_parallel_pipeline(dt_pubmed, cv_folds, run_test_sample = TRUE, test_sample_n_cols = 10)
saveRDS(pubmed_res, file = 'samp_pubmed_results.rds')